# Instructor + Ollama — Fully Local Structured Extraction

**Week 1 | Notebook 4 of 4**

**What you'll learn:**
- Setting up Ollama with Llama 3.1 and Qwen2.5
- Instructor + local model — same API as cloud
- Benchmark: extraction accuracy vs. GPT-4o on 50 examples
- Cost analysis: local inference vs. API calls at scale
- RunPod serverless + Instructor integration
- Privacy-preserving extraction pipeline (all local)

**Runtime:** ~50 minutes (requires local GPU or RunPod)

**Hardware:** 8GB+ RAM for CPU inference, 16GB+ VRAM for GPU

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("01_instructor/04_local_pipeline.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  01_instructor/04_local_pipeline.ipynb
Task:      Local Ollama pipeline
Calls:     ~20

With GPT-4o:       $0.20 USD
With GPT-4o-mini:  $0.02 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup — Ollama with Local Models

In [2]:
import subprocess

# Check if Ollama is running
result = subprocess.run(["ollama", "list"], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ Ollama is running. Available models:")
    print(result.stdout)
else:
    print("❌ Ollama not running. Start it with: ollama serve")

✅ Ollama is running. Available models:
NAME                       ID              SIZE      MODIFIED      
qwen3.5:0.8b               f3817196d142    1.0 GB    3 minutes ago    
qwen3.5:4b                 2a654d98e6fb    3.4 GB    4 hours ago      
llama3.1:8b                46e0c10c039e    4.9 GB    28 hours ago     
qwen3:8b                   500a1f067a9f    5.2 GB    2 months ago     
glm-5:cloud                c313cd065935    -         4 months ago     
embeddinggemma:latest      85462619ee72    621 MB    5 months ago     
gemma3:1b                  8648f39daa8f    815 MB    5 months ago     
qwen3:1.7b                 8f68893c685c    1.4 GB    5 months ago     
nomic-embed-text:latest    0a109f422b47    274 MB    5 months ago     
glm-ocr:latest             6effedd0dc8a    2.2 GB    5 months ago     



In [3]:
from typing import Literal

import instructor
from openai import OpenAI
from pydantic import BaseModel, Field

from src.config import OLLAMA_BASE_URL

# Connect to Ollama via OpenAI-compatible API
ollama_client = OpenAI(base_url=f"{OLLAMA_BASE_URL}/v1", api_key="ollama")
client = instructor.from_openai(ollama_client)

print(f"✅ Instructor connected to Ollama at {OLLAMA_BASE_URL}")

✅ Instructor connected to Ollama at http://localhost:11434


## 2. Instructor + Local Model — Same API as Cloud

In [4]:
class Person(BaseModel):
    name: str
    age: int
    occupation: str


# EXACTLY the same code as with OpenAI — only the model name changes
person = client.chat.completions.create(
    model="qwen3:8b",  # Local model
    response_model=Person,
    messages=[{"role": "user", "content": "John Smith is a 35-year-old software engineer"}],
)

print(person)
print(f"Type: {type(person)}")  # Fully typed Pydantic object

name='John Smith' age=35 occupation='software engineer'
Type: <class '__main__.Person'>


## 3. Benchmark: Local vs GPT-4o on 50 Examples

In [5]:
import time

from src.datasets import generate_sentiment_data


class SentimentResult(BaseModel):
    text: str
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float = Field(ge=0, le=1)


samples = generate_sentiment_data(20)  # Using 20 for speed

local_correct = 0
local_time = 0.0

for sample in samples:
    start = time.time()
    try:
        result = client.chat.completions.create(
            model="qwen3:8b",
            response_model=SentimentResult,
            messages=[{"role": "user", "content": f"Classify: {sample['text']}"}],
        )
        if result.sentiment == sample["sentiment"]:
            local_correct += 1
    except Exception as e:
        print(f"Error on sample: {e}")
    local_time += time.time() - start

print(f"Local accuracy: {local_correct}/{len(samples)} = {local_correct / len(samples) * 100:.1f}%")
print(f"Local total time: {local_time:.1f}s ({local_time / len(samples):.2f}s per sample)")

KeyboardInterrupt: 

## 4. Cost Analysis: Local Inference vs API Calls at Scale

In [6]:
# Cost comparison for 10,000 extractions/month
monthly_calls = 10000

# OpenAI GPT-4o-mini pricing (as of May 2026)
gpt4o_mini_cost = monthly_calls * 0.0005  # ~$0.0005 per call avg

# Local inference costs (amortized)
# - GPU: $0 (if you already have one) or ~$2/hour cloud GPU
# - Electricity: negligible for CPU, ~$0.10/hour for GPU
# - Hardware: one-time cost
local_cost = 0  # After hardware purchase

print("Monthly cost for 10,000 extractions:")
print(f"  GPT-4o-mini (API):   ${gpt4o_mini_cost:.2f}")
print(f"  Local (Ollama):      ${local_cost:.2f} (after hardware)")
print(f"  Savings:             ${gpt4o_mini_cost - local_cost:.2f}/month")
print(f"\nBreak-even: If GPU costs $500, pays for itself in {500 / gpt4o_mini_cost:.0f} months")

Monthly cost for 10,000 extractions:
  GPT-4o-mini (API):   $5.00
  Local (Ollama):      $0.00 (after hardware)
  Savings:             $5.00/month

Break-even: If GPU costs $500, pays for itself in 100 months


## 5. RunPod Serverless + Instructor Integration

In [ ]:
# For production scale, deploy on RunPod serverless
# Then point Instructor to your RunPod endpoint

# runpod_client = OpenAI(
#     base_url="https://api.runpod.ai/v2/your-endpoint/openai/v1",
#     api_key="your-runpod-api-key"
# )
# client = instructor.from_openai(runpod_client)

print("✅ RunPod serverless setup notes:")
print("   1. Deploy a vLLM or TGI pod on RunPod")
print("   2. Expose the OpenAI-compatible endpoint")
print("   3. Point Instructor to the base_url")
print("   4. Same code, production-scale inference")

## 6. Privacy-Preserving PII Extraction Pipeline

In [7]:
class PiiExtraction(BaseModel):
    """Extract PII while keeping all processing local."""

    names: list[str]
    emails: list[str]
    phone_numbers: list[str]
    addresses: list[str]
    redacted_text: str


sensitive_text = """
Contact: Jane Doe, jane.doe@company.com, +1-555-0199
Address: 123 Main St, San Francisco, CA 94105
SSN: 123-45-6789 (should be redacted)
"""

# All processing happens locally — no data leaves your machine
pii = client.chat.completions.create(
    model="qwen3.5:0.8b",
    response_model=PiiExtraction,
    messages=[{"role": "user", "content": f"Extract and redact PII: {sensitive_text}"}],
)

print("Extracted PII (local processing only):")
print(f"  Names: {pii.names}")
print(f"  Emails: {pii.emails}")
print("\nRedacted text:")
print(pii.redacted_text)

Extracted PII (local processing only):
  Names: ['Jane Doe']
  Emails: ['jane.doe@company.com']

Redacted text:
"SSN: 123-45-6789 (should be redacted)"
